In [5]:
import pandas as pd
df = pd.read_csv("../data/resume_job_matching_synthetic_5000.csv")
df.head()
df.shape
df.columns
df.info
df.isnull().sum()
df.duplicated().sum()
df["fit_category"].value_counts()
df["match_score"].describe()

count    5000.000000
mean       59.894400
std        21.691357
min         0.000000
25%        43.000000
50%        60.000000
75%        78.000000
max       100.000000
Name: match_score, dtype: float64

In [6]:
df.duplicated().sum()
df["fit_category"].value_counts()

fit_category
Potential Fit    1817
No Fit           1735
Good Fit         1448
Name: count, dtype: int64

In [11]:
df.loc[0, "resume_text"]
df.loc[0, "job_description"]
df.loc[0, "match_score"]
df.loc[0, "matched_skills"]
df.loc[0, "missing_skills"]

'Docker, Machine Learning'

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
text = df["resume_text"] + " " + df["job_description"]
X = vectorizer.fit_transform(text)
X.shape

(5000, 173)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

resume_tfidf = vectorizer.fit_transform(df["resume_text"])
jd_tfidf = vectorizer.transform(df["job_description"])

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(resume_tfidf, jd_tfidf)

from sklearn.preprocessing import normalize

resume_tfidf = normalize(resume_tfidf)
jd_tfidf = normalize(jd_tfidf)

cosine_scores = resume_tfidf.multiply(jd_tfidf).sum(axis=1)
cosine_scores = cosine_scores.A1

cosine_scores.shape
df["cosine_similarity"] = cosine_scores
df[["match_score", "cosine_similarity"]].head(10)
df["match_score"].corr(df["cosine_similarity"])

np.float64(0.4833375051076145)

In [ ]:
def calculate_skill_overlap(row):
    matched = 0 if pd.isna(row["matched_skills"]) or row["matched_skills"] == "None" else len(row["matched_skills"].split(", "))
    
    missing = 0 if pd.isna(row["missing_skills"]) or row["missing_skills"] == "None" else len(row["missing_skills"].split(", "))
    
    total_required = matched + missing
    
    if total_required == 0:
        return 0
    
    return matched / total_required

In [29]:
df["skill_overlap"] = df.apply(calculate_skill_overlap, axis=1)
df[["match_score", "cosine_similarity", "skill_overlap"]].head(10)

,match_score,cosine_similarity,skill_overlap
0,60,0.702358,0.750000
1,61,0.555714,0.571429
2,17,0.369026,0.142857
3,73,0.711399,0.625000
4,61,0.598581,0.500000
5,65,0.407871,0.600000
6,87,0.729498,0.857143
7,31,0.406487,0.200000
8,92,0.623151,0.875000
9,56,0.525989,0.500000


In [31]:
import re

def extract_candidate_experience(text):
    match = re.search(r'with (\d+) years? of experience', text)
    
    if match:
        return int(match.group(1))
    
    return 0

extract_candidate_experience(df["resume_text"].iloc[0])

0

In [32]:
def extract_required_experience(text):
    match = re.search(r'requires approximately (\d+)\+? years', text)
    
    if match:
        return int(match.group(1))
    
    return 0

extract_required_experience(df["job_description"].iloc[0])

4

In [33]:
df["candidate_experience"] = df["resume_text"].apply(
    extract_candidate_experience
)

df["required_experience"] = df["job_description"].apply(
    extract_required_experience
)

df[[
    "candidate_experience",
    "required_experience"
]].head(10)

,candidate_experience,required_experience
0,0,4
1,6,5
2,2,5
3,4,2
4,8,1
5,6,3
6,6,1
7,3,0
8,4,4
9,6,4


In [35]:
def calculate_experience_match(row):
    candidate = row["candidate_experience"]
    required = row["required_experience"]
    
    if required == 0:
        return 1.0
    
    return min(candidate / required, 1.0)
df["experience_match"] = df.apply(
    calculate_experience_match,
    axis=1
)
df[[
    "candidate_experience",
    "required_experience",
    "experience_match"
]].head(10)

df[[
    "match_score",
    "cosine_similarity",
    "skill_overlap",
    "experience_match"
]].corr()

,match_score,cosine_similarity,skill_overlap,experience_match
match_score,1.000000,0.483338,0.956041,0.264553
cosine_similarity,0.483338,1.000000,0.505976,-0.009127
skill_overlap,0.956041,0.505976,1.000000,-0.009171
experience_match,0.264553,-0.009127,-0.009171,1.000000


In [37]:
features = [
    "cosine_similarity",
    "skill_overlap",
    "experience_match"
]

X = df[features]
y = df["match_score"]

X.head()
y.head()

0    60
1    61
2    17
3    73
4    61
Name: match_score, dtype: int64

In [38]:
#Splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4000, 3)
X_test: (1000, 3)
y_train: (4000,)
y_test: (1000,)


In [ ]:
import xgboost as xgb

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [43]:
y_pred = model.predict(X_test)
print(y_pred[:10])

[48.23516  86.317406 44.520714 58.89636  58.45456  67.107285 66.91485
 37.75416  37.97156  37.93603 ]


In [44]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(10)

,Actual,Predicted
0,46,48.235161
1,89,86.317406
2,43,44.520714
3,61,58.896358
4,61,58.454559
5,69,67.107285
6,68,66.914848
7,35,37.754162
8,40,37.971561
9,37,37.936031


In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 1.9971113204956055
RMSE: 2.3318365836883475
R²  : 0.9883257746696472


In [46]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
1,skill_overlap,0.880139
2,experience_match,0.119284
0,cosine_similarity,0.000577


In [48]:
import joblib

joblib.dump(model, "../models/xgboost_baseline.pkl")
import os

os.path.exists("../models/xgboost_baseline.pkl")

True

In [50]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
sentence = "I have experience building machine learning models using Python."

embedding = embedding_model.encode(sentence)

print(embedding.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17261.89it/s]


(384,)


In [51]:
resume_embeddings = embedding_model.encode(
    df["resume_text"].tolist(),
    show_progress_bar=True
)
resume_embeddings.shape

Batches: 100%|██████████| 157/157 [01:13<00:00,  2.13it/s]


(5000, 384)

In [52]:
jd_embeddings = embedding_model.encode(
    df["job_description"].tolist(),
    show_progress_bar=True
)
jd_embeddings.shape

Batches: 100%|██████████| 157/157 [00:50<00:00,  3.10it/s]


(5000, 384)

In [54]:
from sklearn.preprocessing import normalize

resume_embeddings_norm = normalize(resume_embeddings)
jd_embeddings_norm = normalize(jd_embeddings)

semantic_similarity = (
    resume_embeddings_norm * jd_embeddings_norm
).sum(axis=1)
df["semantic_similarity"] = semantic_similarity
df[[
    "match_score",
    "cosine_similarity",
    "semantic_similarity",
    "skill_overlap",
    "experience_match"
]].head(10)
df["match_score"].corr(df["semantic_similarity"])

np.float64(0.31306773245228003)

In [56]:
features_v2 = [
    "cosine_similarity",
    "semantic_similarity",
    "skill_overlap",
    "experience_match"
]

X_v2 = df[features_v2]
y_v2 = df["match_score"]
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y_v2,
    test_size=0.2,
    random_state=42
)

In [57]:
model_v2 = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model_v2.fit(X_train_v2, y_train_v2)
y_pred_v2 = model_v2.predict(X_test_v2)
mae_v2 = mean_absolute_error(y_test_v2, y_pred_v2)
rmse_v2 = mean_squared_error(y_test_v2, y_pred_v2) ** 0.5
r2_v2 = r2_score(y_test_v2, y_pred_v2)

print("MAE :", mae_v2)
print("RMSE:", rmse_v2)
print("R²  :", r2_v2)

MAE : 2.0011308193206787
RMSE: 2.3377038917763344
R²  : 0.9882669448852539


In [61]:
def predict_match(resume, job_description):
    
    # 1. TF-IDF similarity
    resume_vector = vectorizer.transform([resume])
    jd_vector = vectorizer.transform([job_description])
    
    resume_vector = normalize(resume_vector)
    jd_vector = normalize(jd_vector)
    
    cosine_similarity_score = (
        resume_vector.multiply(jd_vector)
    ).sum()
    
    cosine_similarity_score = cosine_similarity_score.item()
    
    
    # 2. Experience
    candidate_experience = extract_candidate_experience(resume)
    required_experience = extract_required_experience(job_description)
    
    if required_experience == 0:
        experience_match_score = 1.0
    else:
        experience_match_score = min(
            candidate_experience / required_experience,
            1.0
        )
    
    
    # 3. Skill overlap
    # For now we will calculate this from the skills mentioned
    # in our existing dataset logic later.

In [62]:
skills_list = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "TensorFlow",
    "PyTorch",
    "Scikit-learn",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "FastAPI",
    "Flask",
    "Django",
    "Git",
    "Docker",
    "AWS",
    "Azure",
    "GCP",
    "Java",
    "JavaScript",
    "React",
    "Node.js",
    "HTML",
    "CSS",
    "MongoDB",
    "PostgreSQL",
    "MySQL",
    "NLP",
    "Computer Vision",
    "Spark",
    "Hadoop"
]
def extract_skills(text):
    text_lower = text.lower()
    
    found_skills = []
    
    for skill in skills_list:
        if skill.lower() in text_lower:
            found_skills.append(skill)
    
    return found_skills

In [66]:
test_text = """
I have experience with Python, SQL, Pandas,
Machine Learning and AWS.
"""

extract_skills(test_text)
def calculate_new_skill_overlap(resume, job_description):
    
    resume_skills = extract_skills(resume)
    required_skills = extract_skills(job_description)
    
    if len(required_skills) == 0:
        return 0.0
    
    matched_skills = [
        skill for skill in required_skills
        if skill in resume_skills
    ]
    
    skill_overlap = len(matched_skills) / len(required_skills)
    
    return skill_overlap

resume = """
I am a Python developer with experience in Python, SQL,
Pandas and Git.
"""

job_description = """
We are looking for a developer with Python, SQL,
Pandas, AWS and Docker skills.
"""
calculate_new_skill_overlap(resume, job_description)

0.6

In [68]:
def predict_match(resume, job_description):
    
    # 1. Calculate TF-IDF cosine similarity
    resume_vector = vectorizer.transform([resume])
    jd_vector = vectorizer.transform([job_description])
    
    resume_vector = normalize(resume_vector)
    jd_vector = normalize(jd_vector)
    
    cosine_score = (
        resume_vector.multiply(jd_vector)
    ).sum().item()
    
    
    # 2. Calculate skill overlap
    skill_overlap_score = calculate_new_skill_overlap(
        resume,
        job_description
    )
    
    
    # 3. Calculate experience match
    candidate_experience = extract_candidate_experience(resume)
    required_experience = extract_required_experience(job_description)
    
    if required_experience == 0:
        experience_match_score = 1.0
    else:
        experience_match_score = min(
            candidate_experience / required_experience,
            1.0
        )
    
    
    # 4. Create feature DataFrame
    features_for_prediction = pd.DataFrame({
        "cosine_similarity": [cosine_score],
        "skill_overlap": [skill_overlap_score],
        "experience_match": [experience_match_score]
    })
    
    
    # 5. Predict match score
    prediction = model.predict(features_for_prediction)[0]
    
    
    return prediction

resume = """
I am a Python developer with 3 years of experience.
I have experience with Python, SQL, Pandas and Git.
"""

job_description = """
We are looking for a Python developer.
The role requires approximately 4+ years of experience.
Required skills: Python, SQL, Pandas, AWS and Docker.
"""
score = predict_match(resume, job_description)

print("Predicted Match Score:", score)

Predicted Match Score: 63.614185


In [70]:
def predict_match_details(resume, job_description):
    
    # 1. TF-IDF cosine similarity
    resume_vector = vectorizer.transform([resume])
    jd_vector = vectorizer.transform([job_description])
    
    resume_vector = normalize(resume_vector)
    jd_vector = normalize(jd_vector)
    
    cosine_score = (
        resume_vector.multiply(jd_vector)
    ).sum().item()
    
    
    # 2. Extract skills
    resume_skills = extract_skills(resume)
    required_skills = extract_skills(job_description)
    
    matched_skills = [
        skill for skill in required_skills
        if skill in resume_skills
    ]
    
    missing_skills = [
        skill for skill in required_skills
        if skill not in resume_skills
    ]
    
    
    # 3. Skill overlap
    if len(required_skills) == 0:
        skill_overlap_score = 0.0
    else:
        skill_overlap_score = (
            len(matched_skills) / len(required_skills)
        )
    
    
    # 4. Experience
    candidate_experience = extract_candidate_experience(resume)
    required_experience = extract_required_experience(job_description)
    
    if required_experience == 0:
        experience_match_score = 1.0
    else:
        experience_match_score = min(
            candidate_experience / required_experience,
            1.0
        )
    
    
    # 5. Prepare features for XGBoost
    features_for_prediction = pd.DataFrame({
        "cosine_similarity": [cosine_score],
        "skill_overlap": [skill_overlap_score],
        "experience_match": [experience_match_score]
    })
    
    
    # 6. Predict
    prediction = model.predict(features_for_prediction)[0]
    
    
    # 7. Return detailed result
    return {
        "match_score": round(float(prediction), 2),
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "candidate_experience": candidate_experience,
        "required_experience": required_experience
    }

result = predict_match_details(
    resume,
    job_description
)

result

{'match_score': 63.61,
 'matched_skills': ['Python', 'SQL', 'Pandas'],
 'missing_skills': ['Docker', 'AWS'],
 'candidate_experience': 3,
 'required_experience': 4}

In [71]:
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
os.path.exists("../models/tfidf_vectorizer.pkl")

True